# Import libraries
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import json
from datetime import datetime, timedelta

# Configure display
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

print("✓ Libraries imported successfully")

## 1. Load Trade Data

In [1]:
# Load trade data from CSV export
trades_file = 'exports/bot_trades_all.csv'

if Path(trades_file).exists():
    df = pd.read_csv(trades_file)
    df['entry_time'] = pd.to_datetime(df['entry_time'])
    df['exit_time'] = pd.to_datetime(df['exit_time'])
    
    print(f"✓ Loaded {len(df)} trades")
    print(f"  Date range: {df['entry_time'].min()} to {df['exit_time'].max()}")
    print(f"\nColumns: {list(df.columns)}")
else:
    print(f"⚠️  File not found: {trades_file}")
    print("   Run the bot with trades to generate data")
    # Create sample data for demonstration
    df = pd.DataFrame({
        'trade_id': range(1, 51),
        'entry_time': pd.date_range('2026-01-01', periods=50, freq='4H'),
        'exit_time': pd.date_range('2026-01-01 01:00', periods=50, freq='4H'),
        'direction': np.random.choice(['LONG', 'SHORT'], 50),
        'pnl': np.random.randn(50) * 50 + 10,
        'mfe': np.abs(np.random.randn(50) * 80 + 40),
        'mae': np.abs(np.random.randn(50) * 30 + 15),
        'winner_to_loser': np.random.choice([True, False], 50, p=[0.15, 0.85]),
        'runway_utilization': np.random.rand(50) * 1.5,
        'trigger_quality': np.random.choice(['EXCELLENT', 'GOOD', 'POOR'], 50, p=[0.3, 0.5, 0.2]),
        'harvester_quality': np.random.choice(['EXCELLENT', 'GOOD', 'POOR'], 50, p=[0.4, 0.4, 0.2]),
    })
    print("✓ Generated sample data for demonstration")

# Display first few trades
df.head()

NameError: name 'Path' is not defined

## 2. Performance Summary

In [2]:
# Calculate key metrics
total_trades = len(df)
winners = (df['pnl'] > 0).sum()
losers = (df['pnl'] < 0).sum()
win_rate = winners / total_trades if total_trades > 0 else 0

total_pnl = df['pnl'].sum()
avg_win = df[df['pnl'] > 0]['pnl'].mean() if winners > 0 else 0
avg_loss = df[df['pnl'] < 0]['pnl'].mean() if losers > 0 else 0

# Sharpe ratio (assuming daily returns)
returns = df['pnl'].values
sharpe = (returns.mean() / returns.std() * np.sqrt(252)) if returns.std() > 0 else 0

# Max drawdown
cum_pnl = df['pnl'].cumsum()
running_max = cum_pnl.cummax()
drawdown = (cum_pnl - running_max) / running_max.where(running_max != 0, 1)
max_dd = drawdown.min()

# Winner-to-loser rate
wtl_rate = df['winner_to_loser'].sum() / total_trades if total_trades > 0 else 0

# Display summary
print("╔" + "═" * 58 + "╗")
print("║" + " " * 15 + "PERFORMANCE SUMMARY" + " " * 24 + "║")
print("╚" + "═" * 58 + "╝")
print(f"\nTotal Trades:       {total_trades}")
print(f"Winners:            {winners} ({win_rate*100:.1f}%)")
print(f"Losers:             {losers}")
print(f"\nTotal PnL:          ${total_pnl:,.2f}")
print(f"Average Win:        ${avg_win:.2f}")
print(f"Average Loss:       ${avg_loss:.2f}")
print(f"Profit Factor:      {abs(avg_win * winners / (avg_loss * losers)) if losers > 0 and avg_loss != 0 else 0:.2f}")
print(f"\nSharpe Ratio:       {sharpe:.3f}")
print(f"Max Drawdown:       {max_dd*100:.2f}%")
print(f"Winner→Loser Rate:  {wtl_rate*100:.1f}%")

NameError: name 'df' is not defined

## 3. PnL Curve & Drawdown

In [3]:
# Create PnL curve with drawdown
df['cumulative_pnl'] = df['pnl'].cumsum()
df['running_max'] = df['cumulative_pnl'].cummax()
df['drawdown'] = (df['cumulative_pnl'] - df['running_max']) / df['running_max'].where(df['running_max'] != 0, 1)

fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.7, 0.3],
    subplot_titles=('Cumulative PnL', 'Drawdown'),
    vertical_spacing=0.1
)

# PnL curve
fig.add_trace(
    go.Scatter(
        x=df['exit_time'],
        y=df['cumulative_pnl'],
        mode='lines',
        name='Cumulative PnL',
        line=dict(color='blue', width=2),
        fill='tozeroy',
        fillcolor='rgba(0, 100, 255, 0.1)'
    ),
    row=1, col=1
)

# Drawdown
fig.add_trace(
    go.Scatter(
        x=df['exit_time'],
        y=df['drawdown'] * 100,
        mode='lines',
        name='Drawdown',
        line=dict(color='red', width=2),
        fill='tozeroy',
        fillcolor='rgba(255, 0, 0, 0.2)'
    ),
    row=2, col=1
)

fig.update_xaxes(title_text="Date", row=2, col=1)
fig.update_yaxes(title_text="PnL ($)", row=1, col=1)
fig.update_yaxes(title_text="Drawdown (%)", row=2, col=1)

fig.update_layout(
    height=600,
    title_text="Performance Over Time",
    showlegend=True,
    hovermode='x unified'
)

fig.show()

NameError: name 'df' is not defined

## 4. Win/Loss Distribution

In [4]:
# Histogram of PnL distribution
fig = go.Figure()

# Winners
winners_df = df[df['pnl'] > 0]
fig.add_trace(go.Histogram(
    x=winners_df['pnl'],
    name='Winners',
    marker_color='green',
    opacity=0.7,
    nbinsx=20
))

# Losers
losers_df = df[df['pnl'] < 0]
fig.add_trace(go.Histogram(
    x=losers_df['pnl'],
    name='Losers',
    marker_color='red',
    opacity=0.7,
    nbinsx=20
))

fig.update_layout(
    title='PnL Distribution',
    xaxis_title='PnL ($)',
    yaxis_title='Frequency',
    barmode='overlay',
    height=400
)

fig.show()

NameError: name 'go' is not defined

## 5. MFE/MAE Analysis (Path Efficiency)

In [5]:
# MFE vs MAE scatter plot
df['capture_ratio'] = df['pnl'] / df['mfe'].where(df['mfe'] > 0, 1)
df['trade_outcome'] = df['pnl'].apply(lambda x: 'Winner' if x > 0 else 'Loser')

fig = px.scatter(
    df,
    x='mae',
    y='mfe',
    color='trade_outcome',
    size='pnl',
    hover_data=['trade_id', 'pnl', 'capture_ratio', 'winner_to_loser'],
    title='MFE vs MAE (Trade Path Efficiency)',
    labels={'mae': 'MAE (Max Adverse Excursion)', 'mfe': 'MFE (Max Favorable Excursion)'},
    color_discrete_map={'Winner': 'green', 'Loser': 'red'},
    height=500
)

# Add diagonal line (perfect capture)
max_val = max(df['mae'].max(), df['mfe'].max())
fig.add_trace(go.Scatter(
    x=[0, max_val],
    y=[0, max_val],
    mode='lines',
    name='Perfect Capture',
    line=dict(color='gray', dash='dash'),
    showlegend=True
))

fig.show()

# Winner-to-loser trades (critical failures)
wtl_trades = df[df['winner_to_loser'] == True]
print(f"\n⚠️  Winner-to-Loser Trades: {len(wtl_trades)} ({len(wtl_trades)/len(df)*100:.1f}%)")
if len(wtl_trades) > 0:
    print(f"   Avg MFE: ${wtl_trades['mfe'].mean():.2f}")
    print(f"   Avg Final PnL: ${wtl_trades['pnl'].mean():.2f}")
    print(f"   Avg Capture: {wtl_trades['capture_ratio'].mean()*100:.1f}%")

NameError: name 'df' is not defined

## 6. Dual-Agent Attribution Analysis

In [ ]:
# Agent quality comparison
if 'trigger_quality' in df.columns and 'harvester_quality' in df.columns:
    
    # TriggerAgent performance by quality
    trigger_stats = df.groupby('trigger_quality').agg({
        'pnl': ['count', 'mean', 'sum'],
        'runway_utilization': 'mean',
        'mfe': 'mean'
    }).round(2)
    
    print("TriggerAgent Performance by Quality:")
    print(trigger_stats)
    print()
    
    # HarvesterAgent performance by quality
    harvester_stats = df.groupby('harvester_quality').agg({
        'pnl': ['count', 'mean', 'sum'],
        'capture_ratio': 'mean',
        'winner_to_loser': 'sum'
    }).round(2)
    
    print("HarvesterAgent Performance by Quality:")
    print(harvester_stats)
    
    # Visualize quality distribution
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('TriggerAgent Quality', 'HarvesterAgent Quality'),
        specs=[[{'type': 'pie'}, {'type': 'pie'}]]
    )
    
    # TriggerAgent pie
    trigger_counts = df['trigger_quality'].value_counts()
    fig.add_trace(go.Pie(
        labels=trigger_counts.index,
        values=trigger_counts.values,
        marker=dict(colors=['green', 'yellow', 'red']),
        name='Trigger'
    ), row=1, col=1)
    
    # HarvesterAgent pie
    harvester_counts = df['harvester_quality'].value_counts()
    fig.add_trace(go.Pie(
        labels=harvester_counts.index,
        values=harvester_counts.values,
        marker=dict(colors=['green', 'yellow', 'red']),
        name='Harvester'
    ), row=1, col=2)
    
    fig.update_layout(height=400, title_text="Agent Quality Distribution")
    fig.show()
    
else:
    print("⚠️  Dual-agent attribution data not available")

## 7. Runway Prediction Accuracy (TriggerAgent)

In [ ]:
# Runway utilization analysis
if 'runway_utilization' in df.columns:
    
    fig = go.Figure()
    
    fig.add_trace(go.Histogram(
        x=df['runway_utilization'],
        nbinsx=30,
        marker_color='blue',
        opacity=0.7,
        name='Runway Utilization'
    ))
    
    # Add vertical line at 100% (perfect prediction)
    fig.add_vline(
        x=1.0,
        line_dash="dash",
        line_color="green",
        annotation_text="Perfect (100%)",
        annotation_position="top"
    )
    
    fig.update_layout(
        title='TriggerAgent Runway Prediction Accuracy',
        xaxis_title='Runway Utilization (actual MFE / predicted)',
        yaxis_title='Frequency',
        height=400
    )
    
    fig.show()
    
    # Stats
    print(f"\nRunway Utilization Statistics:")
    print(f"  Mean:   {df['runway_utilization'].mean():.2f}")
    print(f"  Median: {df['runway_utilization'].median():.2f}")
    print(f"  Std:    {df['runway_utilization'].std():.2f}")
    
    # Accuracy bands
    excellent = ((df['runway_utilization'] >= 0.8) & (df['runway_utilization'] <= 1.2)).sum()
    good = ((df['runway_utilization'] >= 0.6) & (df['runway_utilization'] < 0.8) | 
            (df['runway_utilization'] > 1.2) & (df['runway_utilization'] <= 1.4)).sum()
    poor = ((df['runway_utilization'] < 0.6) | (df['runway_utilization'] > 1.4)).sum()
    
    print(f"\nPrediction Accuracy:")
    print(f"  Excellent (±20%): {excellent} ({excellent/len(df)*100:.1f}%)")
    print(f"  Good (±40%):      {good} ({good/len(df)*100:.1f}%)")
    print(f"  Poor (>40%):      {poor} ({poor/len(df)*100:.1f}%)")

else:
    print("⚠️  Runway utilization data not available")

## 8. Trade Direction Analysis

In [ ]:
# Performance by direction
direction_stats = df.groupby('direction').agg({
    'pnl': ['count', 'sum', 'mean'],
    'mfe': 'mean',
    'mae': 'mean'
})

print("Performance by Direction:")
print(direction_stats)
print()

# Win rate by direction
direction_wins = df[df['pnl'] > 0].groupby('direction').size()
direction_total = df.groupby('direction').size()
direction_wr = (direction_wins / direction_total * 100).round(1)

print("Win Rate by Direction:")
for dir in direction_wr.index:
    print(f"  {dir}: {direction_wr[dir]:.1f}%")

## 9. Export Summary Report

In [ ]:
# Generate summary report
report = {
    'analysis_date': datetime.now().isoformat(),
    'total_trades': int(total_trades),
    'win_rate': float(win_rate),
    'total_pnl': float(total_pnl),
    'sharpe_ratio': float(sharpe),
    'max_drawdown': float(max_dd),
    'avg_win': float(avg_win),
    'avg_loss': float(avg_loss),
    'wtl_rate': float(wtl_rate),
    'best_trade': float(df['pnl'].max()),
    'worst_trade': float(df['pnl'].min()),
}

# Save report
report_file = 'exports/analysis_report.json'
Path('exports').mkdir(exist_ok=True)
with open(report_file, 'w') as f:
    json.dump(report, f, indent=2)

print(f"✓ Summary report saved to: {report_file}")
print("\n" + json.dumps(report, indent=2))